In [1]:
import pickle

import numpy as np
import pandas as pd
import faiss
import torch
from torch.utils.data import DataLoader

from utils import *

In [2]:
interactions = pd.read_csv(
    "data/evaluate.csv",
    names=['USER_ID', 'VIDEO_ID', 'INTERACTION_TYPE', 'TIMESTAMP'],
    header=None
)

In [3]:
interactions = interactions.dropna(
    subset=['USER_ID', 'VIDEO_ID', 'INTERACTION_TYPE', 'TIMESTAMP']
)

In [4]:
interactions['USER_ID'] = interactions['USER_ID'].astype("Int64")
interactions['VIDEO_ID'] = interactions['VIDEO_ID'].astype("Int64")
interactions['INTERACTION_TYPE'] = interactions['INTERACTION_TYPE'].astype("Int64")
interactions['TIMESTAMP'] = pd.to_datetime(interactions['TIMESTAMP'], unit="s", utc=True)

In [5]:
pd.concat([interactions.head(), interactions.tail()])

,USER_ID,VIDEO_ID,INTERACTION_TYPE,TIMESTAMP
0,1,46763,0,2017-01-09 19:50:16+00:00
1,1,173483,0,2017-01-09 19:47:42+00:00
2,1,284109,0,2017-01-09 19:50:16+00:00
3,1,331777,0,2017-01-09 19:50:16+00:00
4,1,379208,0,2017-01-09 19:47:42+00:00
19433732,1483626,34633,5,2017-01-24 13:00:24+00:00
19433733,1483626,91594,5,2017-01-24 13:00:24+00:00
19433734,1485180,640382,5,2016-12-27 14:59:44+00:00
19433735,1490693,124350,5,2017-01-18 07:01:59+00:00
19433736,1490693,472666,5,2017-01-18 07:01:59+00:00


In [6]:
interactions.shape

(19433737, 4)

In [7]:
interactions = interactions[interactions['INTERACTION_TYPE'].isin([1, 2, 3, 5])]

interactions = interactions.drop(['INTERACTION_TYPE'], axis=1)

In [8]:
interactions.shape

(3729257, 3)

In [9]:
eval_interactions = interactions.copy()

In [10]:
eval_interactions.shape

(3729257, 3)

In [11]:
eval_interactions = eval_interactions.sort_values(['USER_ID', 'TIMESTAMP'])

eval_interactions['VIDEO_ID_SEQ'] = eval_interactions.groupby(['USER_ID'])['VIDEO_ID'].transform(
    lambda video_id_series: [
        video_id_series.iloc[:idx].tolist()[-MAX_SEQ_LEN:]
        for idx in range(len(video_id_series))
    ]
)

In [12]:
pd.concat([eval_interactions.head(), eval_interactions.tail()])

,USER_ID,VIDEO_ID,TIMESTAMP,VIDEO_ID_SEQ
15704480,3,726510,2017-01-02 07:21:25+00:00,[]
15704490,4,360712,2016-11-07 19:51:07+00:00,[]
15704492,4,417897,2016-11-10 15:21:21+00:00,[360712]
15704499,4,743981,2016-11-15 11:52:01+00:00,"[360712, 417897]"
15704485,4,171061,2016-11-16 10:01:06+00:00,"[360712, 417897, 743981]"
19193610,1497014,820673,2017-01-24 08:44:27+00:00,"[1296299, 131352, 1293024, 55677, 389888, 7566..."
19193603,1497014,425607,2017-01-24 22:06:44+00:00,"[1296299, 131352, 1293024, 55677, 389888, 7566..."
19193618,1497015,273817,2017-01-05 13:12:34+00:00,[]
19193620,1497018,1003535,2016-11-17 09:01:53+00:00,[]
19193619,1497018,789481,2017-01-19 12:33:53+00:00,[1003535]


In [13]:
eval_interactions.shape

(3729257, 4)

In [14]:
with open(VIDEO_ID_TO_VIDEO_IDX_PATH, "rb") as file:
    video_id_to_video_idx = pickle.load(file)

In [15]:
eval_interactions['VIDEO_IDX'] = eval_interactions['VIDEO_ID'].map(
    lambda video_id:
    video_id_to_video_idx.get(str(video_id), UNK_IDX)
)
eval_interactions = eval_interactions.drop(['VIDEO_ID'], axis=1)

eval_interactions['VIDEO_IDX_SEQ'] = eval_interactions['VIDEO_ID_SEQ'].map(
    lambda video_id_seq: [
        video_id_to_video_idx.get(str(video_id), UNK_IDX)
        for video_id in video_id_seq
    ] if video_id_seq else [BGN_IDX]
)
eval_interactions = eval_interactions.drop(['VIDEO_ID_SEQ'], axis=1)

eval_interactions['VIDEO_IDX_SEQ_LEN'] = eval_interactions['VIDEO_IDX_SEQ'].map(
    lambda video_idx_seq:
    len(video_idx_seq)
)

In [16]:
target_video_idxs = eval_interactions['VIDEO_IDX'].values

In [17]:
pd.concat([eval_interactions.head(), eval_interactions.tail()])

,USER_ID,TIMESTAMP,VIDEO_IDX,VIDEO_IDX_SEQ,VIDEO_IDX_SEQ_LEN
15704480,3,2017-01-02 07:21:25+00:00,186188,[2],1
15704490,4,2016-11-07 19:51:07+00:00,92008,[2],1
15704492,4,2016-11-10 15:21:21+00:00,106896,[92008],1
15704499,4,2016-11-15 11:52:01+00:00,190558,"[92008, 106896]",2
15704485,4,2016-11-16 10:01:06+00:00,43560,"[92008, 106896, 190558]",3
19193610,1497014,2017-01-24 08:44:27+00:00,210379,"[332466, 33354, 331673, 14076, 99552, 193902, ...",22
19193603,1497014,2017-01-24 22:06:44+00:00,108872,"[332466, 33354, 331673, 14076, 99552, 193902, ...",23
19193618,1497015,2017-01-05 13:12:34+00:00,69880,[2],1
19193620,1497018,2016-11-17 09:01:53+00:00,257445,[2],1
19193619,1497018,2017-01-19 12:33:53+00:00,202244,[257445],1


In [18]:
eval_interactions.shape

(3729257, 5)

In [19]:
eval_dataset = DatasetForEvaluation(
    interactions=eval_interactions
)

In [20]:
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed(RANDOM_STATE)

In [21]:
eval_dataloader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=evaluation_collate_fn,
    pin_memory=True,
    persistent_workers=True
)

In [22]:
with open(USER_ENCODER_KWARGS_PATH, "rb") as file:
    user_encoder_kwargs = pickle.load(file)

In [23]:
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed(RANDOM_STATE)

In [24]:
user_encoder = UserEncoder(**user_encoder_kwargs)

In [25]:
print(f"user encoder parameters: {sum(parameters.numel() for parameters in user_encoder.parameters())}")

user encoder parameters: 43114240


In [26]:
user_encoder.load_state_dict(torch.load(USER_ENCODER_PATH))

<All keys matched successfully>

In [27]:
user_encoder = user_encoder.to(DEVICE)

user_embeddings = []

user_encoder.eval()

with torch.inference_mode():

    for batch in eval_dataloader:

        video_idx_seq = batch["video_idx_seq"].to(DEVICE, non_blocking=True)
        video_idx_seq_len = batch["video_idx_seq_len"].to(DEVICE, non_blocking=True)

        user_embedding = user_encoder(
            video_idx_seq,
            video_idx_seq_len
        )

        user_embeddings.extend(
            user_embedding.cpu().numpy()
        )

    user_embeddings = np.array(user_embeddings).astype("float32")

In [28]:
video_embeddings_index = faiss.read_index(VIDEO_EMBEDDINGS_INDEX_PATH)

In [29]:
samples_count = user_embeddings.shape[0]

eval_hit_10_count = 0
eval_hit_100_count = 0
eval_hit_1000_count = 0

for idx in range(0, samples_count, BATCH_SIZE):

    user_embeddings_batch = user_embeddings[idx:idx + BATCH_SIZE]
    target_video_idxs_batch = target_video_idxs[idx:idx + BATCH_SIZE]

    _, eval_top_1000 = video_embeddings_index.search(
        user_embeddings_batch,
        1000
    )

    eval_hit_10_count += np.any(
        eval_top_1000[:, :10] == target_video_idxs_batch[:, None],
        axis=1
    ).sum()

    eval_hit_100_count += np.any(
        eval_top_1000[:, :100] == target_video_idxs_batch[:, None],
        axis=1
    ).sum()

    eval_hit_1000_count += np.any(
        eval_top_1000 == target_video_idxs_batch[:, None],
        axis=1
    ).sum()

eval_hit_10_rate = eval_hit_10_count / samples_count
eval_hit_100_rate = eval_hit_100_count / samples_count
eval_hit_1000_rate = eval_hit_1000_count / samples_count

print(f"eval hit@10: {eval_hit_10_rate:.2f}")
print(f"eval hit@100: {eval_hit_100_rate:.2f}")
print(f"eval hit@1000: {eval_hit_1000_rate:.2f}")

eval hit@10: 0.47
eval hit@100: 0.57
eval hit@1000: 0.69
